# Moon Surface Mosaic Assembler

## Purpose
Stitches all 15,935 Marius Hills NPZ tiles into a single full-resolution mosaic of the lunar surface, without any YOLO overlays.

## How the tiles are organized
- Each NPZ file contains a `(3, 256, 256)` float32 image (3 RGB channels, values 0–1).
- Tiles are named `marius_hills_r{row5}_c{col5}.npz` (5-digit zero-padded row/col).
- The row/col in the filename is a **tile index**, not a pixel coordinate.
- Tiles overlap by 50%: each tile is 256×256 but placed with a stride of 128 pixels. This is typical for sliding-window inference where edge predictions are discarded but here we keep the full tile.

## Step-by-step

### 1. Grid discovery (lines 10–21)
Scan all filenames to find the maximum tile row and column indices. The full mosaic dimensions are:
```
Height = max_row × 128 + 256
Width  = max_col × 128 + 256
```

### 2. Accumulation buffers (lines 24–25)
Two arrays sized to the full mosaic:
- **`accum`** (`float64`, H×W×3) — sums all tile pixel values at each position
- **`counts`** (`int32`, H×W) — counts how many tiles cover each pixel

Using `float64` prevents precision loss during summation of up to 4 overlapping tiles per pixel.

### 3. Tile placement loop (lines 27–39)
For each NPZ file:
1. Extract the tile row/col index from the filename.
2. Compute the top-left pixel position: `(row × 128, col × 128)`.
3. Load the image, transpose from `(3, 256, 256)` to `(256, 256, 3)`.
4. Add the tile into `accum` at the correct position and increment `counts`.

### 4. Averaging & output (lines 42–45)
Divide each pixel's accumulated sum by the number of tiles covering it (handles 2×2 tile overlaps in interior regions, 1 tile at corners, 2 at edges). Clip to [0,1], scale to uint8 [0,255], and save as PNG.

## Output
- `/home/podman/yolo_moon/moon_mosaic.png` — full lunar surface mosaic at native resolution (~11,520×23,040 pixels, ~800 MB PNG).


In [2]:
import numpy as np
from glob import glob
import re, os
from PIL import Image

NPZ_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/data/MR/data/processed/tiles/marius_hills'
OUT_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images'
OUT_FILE = f'{OUT_DIR}/moon_mosaic.png'

files = sorted(glob(f'{NPZ_DIR}/marius_hills_*.npz'))
print(f'Total tiles: {len(files)}')

# Determine mosaic extent from NPZ row/col values
max_r = max_c = 0
for fpath in files:
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    max_r = max(max_r, ridx)
    max_c = max(max_c, cidx)

# Each tile is 256x256, stride is 128 (50% overlap)
# Full extent = last_tile_index * 128 + 256
TILE_SIZE = 256
STRIDE = 128
H = max_r * STRIDE + TILE_SIZE
W = max_c * STRIDE + TILE_SIZE
print(f'Grid: {max_r+1}x{max_c+1} tiles, Mosaic: {H}x{W}')

# Accumulate with float64 for precision, then average
accum = np.zeros((H, W, 3), dtype=np.float64)
counts = np.zeros((H, W), dtype=np.int32)

for i, fpath in enumerate(files):
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    r0, c0 = ridx * STRIDE, cidx * STRIDE

    img_np = np.load(fpath)['image']           # (3, 256, 256) float32, 0-1
    img_rgb = np.transpose(img_np, (1, 2, 0))  # (256, 256, 3)

    accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE] += img_rgb
    counts[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE] += 1

    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1}/{len(files)}')

# Average overlapping regions
mosaic = np.where(counts[..., None] > 0, accum / counts[..., None], 0)
mosaic_u8 = (np.clip(mosaic, 0, 1) * 255).astype(np.uint8)

Image.fromarray(mosaic_u8).save(OUT_FILE)
print(f'Saved {OUT_FILE} ({W}x{H})')


Total tiles: 15935
Grid: 89x179 tiles, Mosaic: 11520x23040
  Processed 1000/15935
  Processed 2000/15935
  Processed 3000/15935
  Processed 4000/15935
  Processed 5000/15935
  Processed 6000/15935
  Processed 7000/15935
  Processed 8000/15935
  Processed 9000/15935
  Processed 10000/15935
  Processed 11000/15935
  Processed 12000/15935
  Processed 13000/15935
  Processed 14000/15935
  Processed 15000/15935
Saved /home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images/moon_mosaic.png (23040x11520)


In [1]:
import numpy as np
from glob import glob
import re, os
from PIL import Image

NPZ_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/data/MR/data/processed/tiles/marius_hills'
OUT_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images'
OUT_FILE = f'{OUT_DIR}/moon_mosaic.png'

files = sorted(glob(f'{NPZ_DIR}/marius_hills_*.npz'))
print(f'Total tiles: {len(files)}')

# Determine mosaic extent from NPZ row/col values
max_r = max_c = 0
for fpath in files:
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    max_r = max(max_r, ridx)
    max_c = max(max_c, cidx)

# Each tile is 256x256, stride is 128 (50% overlap)
# Full extent = last_tile_index * 128 + 256
TILE_SIZE = 256
STRIDE = 128
H = max_r * STRIDE + TILE_SIZE
W = max_c * STRIDE + TILE_SIZE
print(f'Grid: {max_r+1}x{max_c+1} tiles, Mosaic: {H}x{W}')

# Accumulate with float64 for precision, then average
accum = np.zeros((H, W, 3), dtype=np.float64)
counts = np.zeros((H, W), dtype=np.int32)

for i, fpath in enumerate(files):
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    r0, c0 = ridx * STRIDE, cidx * STRIDE

    img_np = np.load(fpath)['mask'][0:1,:,:]           # (3, 256, 256) float32, 0-1
    img_rgb = np.transpose(img_np, (1, 2, 0))  # (256, 256, 3)

    accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE] += img_rgb
    counts[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE] += 1

    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1}/{len(files)}')

# Average overlapping regions
mosaic = np.where(counts[..., None] > 0, accum / counts[..., None], 0)
mosaic_u8 = (np.clip(mosaic, 0, 1) * 255).astype(np.uint8)

Image.fromarray(mosaic_u8).save(OUT_FILE)
print(f'Saved {OUT_FILE} ({W}x{H})')


Total tiles: 15935
Grid: 89x179 tiles, Mosaic: 11520x23040
  Processed 1000/15935
  Processed 2000/15935
  Processed 3000/15935
  Processed 4000/15935
  Processed 5000/15935
  Processed 6000/15935
  Processed 7000/15935
  Processed 8000/15935
  Processed 9000/15935
  Processed 10000/15935
  Processed 11000/15935
  Processed 12000/15935
  Processed 13000/15935
  Processed 14000/15935
  Processed 15000/15935
Saved /home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images/moon_mosaic.png (23040x11520)


In [1]:
import numpy as np
from glob import glob
import re, os
from PIL import Image
from scipy.ndimage import label

NPZ_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/data/MR/data/processed/tiles/marius_hills'
OUT_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images'
OUT_FILE = f'{OUT_DIR}/moon_mosaic_overlay.png'

files = sorted(glob(f'{NPZ_DIR}/marius_hills_*.npz'))
print(f'Total tiles: {len(files)}')

# 1. Determine mosaic global extent from NPZ row/col values
max_r = max_c = 0
for fpath in files:
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    max_r = max(max_r, ridx)
    max_c = max(max_c, cidx)

TILE_SIZE = 256
STRIDE = 128
H = max_r * STRIDE + TILE_SIZE
W = max_c * STRIDE + TILE_SIZE
print(f'Grid: {max_r+1}x{max_c+1} tiles, Mosaic: {H}x{W}')

# 2. Setup accumulators for both the base image and the mask colors
img_accum = np.zeros((H, W, 3), dtype=np.float64)
mask_accum = np.zeros((H, W, 3), dtype=np.float64)
counts = np.zeros((H, W, 3), dtype=np.int32)

# Color palette for the 7 classes (RGB ratios)
# COLORS = [
#     [1.0, 0.0, 0.0],  # Class 0: Red
#     [0.0, 1.0, 0.0],  # Class 1: Green
#     [0.0, 0.0, 1.0],  # Class 2: Blue
#     [1.0, 1.0, 0.0],  # Class 3: Yellow
#     [1.0, 0.0, 1.0],  # Class 4: Magenta
#     [0.0, 1.0, 1.0],  # Class 5: Cyan
#     [1.0, 0.5, 0.0]   # Class 6: Orange
# ]


COLORS = [
    (255, 0, 0, 180),      # 0: Red
    (0, 255, 0, 180),      # 1: Green
    (0, 0, 255, 180),      # 2: Blue
    (255, 255, 0, 180),    # 3: Yellow
    (0, 255, 255, 180),    # 4: Cyan
    (255, 0, 255, 180),    # 5: Magenta
    (255, 165, 0, 180),    # 6: Orange
]

# 3. Process each tile
for i, fpath in enumerate(files):
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    r0, c0 = ridx * STRIDE, cidx * STRIDE

    # Load original image and masks
    data = np.load(fpath)
    img_np = data['image']  # Shape: (3, 256, 256) or (1, 256, 256)
    masks = data['mask']    # Shape: (7, 256, 256)
    
    # Handle image channel alignment
    if img_np.shape[0] == 1:
        # Convert grayscale (1, 256, 256) to RGB (3, 256, 256) by repeating channels
        img_np = np.repeat(img_np, 3, axis=0)
    
    # Convert image to (256, 256, 3) float64
    img_rgb = np.transpose(img_np, (1, 2, 0)).astype(np.float64)
    # Normalize image to [0, 1] range if it's in 0-255 format
    if img_rgb.max() > 1.0:
        img_rgb /= 255.0

    # Initialize a mask color canvas for just this tile
    tile_mask_rgb = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.float64)

    for class_id in range(7):
        single_mask = masks[class_id] > 0
        th = 100 if class_id == 0 else 300
        lth = 5

        # Label connected components
        labeled, num = label(single_mask)
        filtered_mask = np.zeros_like(single_mask, dtype=np.float64)
        
        for comp_id in range(1, num + 1):
            ys, xs = np.where(labeled == comp_id)
            if len(xs) == 0:
                continue

            box_w = xs.max() - xs.min() + 1
            box_h = ys.max() - ys.min() + 1

            if (box_w < th and box_h < th) and (box_w > lth and box_h > lth):
                filtered_mask[labeled == comp_id] = 1.0

        # Map filtered mask to assigned class colors
        for channel in range(3):
            tile_mask_rgb[..., channel] += filtered_mask * COLORS[class_id][channel]

    # Accumulate data into global canvas layers
    img_accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += img_rgb
    mask_accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += tile_mask_rgb
    counts[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += 1

    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1}/{len(files)}')

# 4. Normalize and combine the layers
# Average out overlap seams for both base image and mask colors
final_img = np.where(counts > 0, img_accum / counts, 0.0)
final_masks = np.where(counts > 0, mask_accum / counts, 0.0)

# 5. Blend the background moon image with the masks
# ALPHA dictates mask opacity (e.g., 0.4 means 40% mask color, 60% moon background)
ALPHA = 0.4
has_mask = np.any(final_masks > 0, axis=-1, keepdims=True)

# Blend only where masks exist, leave background untouched elsewhere
mosaic = np.where(has_mask, (ALPHA * final_masks) + ((1 - ALPHA) * final_img), final_img)
mosaic_u8 = (np.clip(mosaic, 0, 1) * 255).astype(np.uint8)

os.makedirs(OUT_DIR, exist_ok=True)
Image.fromarray(mosaic_u8).save(OUT_FILE)
print(f'Saved {OUT_FILE} ({W}x{H})')

Total tiles: 15935
Grid: 89x179 tiles, Mosaic: 11520x23040
  Processed 1000/15935
  Processed 2000/15935
  Processed 3000/15935
  Processed 4000/15935
  Processed 5000/15935
  Processed 6000/15935
  Processed 7000/15935
  Processed 8000/15935
  Processed 9000/15935
  Processed 10000/15935
  Processed 11000/15935
  Processed 12000/15935
  Processed 13000/15935
  Processed 14000/15935
  Processed 15000/15935
Saved /home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images/moon_mosaic_overlay.png (23040x11520)


In [2]:
1+1

2

In [5]:
import numpy as np
from glob import glob
import re, os
from PIL import Image
from scipy.ndimage import label

NPZ_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/data/MR/data/processed/tiles/marius_hills'
OUT_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images'
OUT_FILE = f'{OUT_DIR}/moon_mosaic_rgba_overlay.png'

files = sorted(glob(f'{NPZ_DIR}/marius_hills_*.npz'))
print(f'Total tiles: {len(files)}')

# 1. Determine mosaic global extent from NPZ row/col values
max_r = max_c = 0
for fpath in files:
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    max_r = max(max_r, ridx)
    max_c = max(max_c, cidx)

TILE_SIZE = 256
STRIDE = 128
H = max_r * STRIDE + TILE_SIZE
W = max_c * STRIDE + TILE_SIZE
print(f'Grid: {max_r+1}x{max_c+1} tiles, Mosaic: {H}x{W}')

# Your exact requested color palette (RGBA format)
a = 280
mask_colors = [
    (255, 0, 0, a),      # 0: Red
    (0, 255, 0, a),      # 1: Green
    (0, 0, 255, a),      # 2: Blue
    (255, 255, 0, a),    # 3: Yellow
    (0, 255, 255, a),    # 4: Cyan
    (255, 0, 255, a),    # 5: Magenta
    (255, 165, 0, a),    # 6: Orange
]

# Extract normalized color vectors and a constant alpha scalar
COLORS = [[c[0]/255.0, c[1]/255.0, c[2]/255.0] for c in mask_colors]
ALPHA = mask_colors[0][3] / 255.0  # Uses 180/255 -> ~0.706 opacity

# 2. Setup accumulators
img_accum = np.zeros((H, W, 3), dtype=np.float64)
mask_accum = np.zeros((H, W, 3), dtype=np.float64)
counts = np.zeros((H, W, 3), dtype=np.int32)

# 3. Process each tile
for i, fpath in enumerate(files):
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    r0, c0 = ridx * STRIDE, cidx * STRIDE

    # Load image and masks
    data = np.load(fpath)
    img_np = data['image']  # Shape: (3, 256, 256)
    masks = data['mask']    # Shape: (7, 256, 256)
    
    # Transpose from (3, 256, 256) to (256, 256, 3)
    img_rgb = np.transpose(img_np, (1, 2, 0)).astype(np.float64)
    if img_rgb.max() > 1.0:
        img_rgb /= 255.0

    # Convert RGB background image to Grayscale via luminosity formula
    gray_channel = np.dot(img_rgb[..., :3], [0.299, 0.587, 0.114])
    img_gray_rgb = np.stack([gray_channel] * 3, axis=-1)

    # Initialize a mask color canvas for this tile
    tile_mask_rgb = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.float64)

    for class_id in range(7):
        single_mask = masks[class_id] > 0
        th = 100 if class_id == 0 else 300
        lth = 5

        # Label connected components
        labeled, num = label(single_mask)
        filtered_mask = np.zeros_like(single_mask, dtype=np.float64)
        
        for comp_id in range(1, num + 1):
            ys, xs = np.where(labeled == comp_id)
            if len(xs) == 0:
                continue

            box_w = xs.max() - xs.min() + 1
            box_h = ys.max() - ys.min() + 1

            if (box_w < th and box_h < th) and (box_w > lth and box_h > lth):
                filtered_mask[labeled == comp_id] = 1.0

        # Map filtered mask to assigned class colors
        for channel in range(3):
            tile_mask_rgb[..., channel] += filtered_mask * COLORS[class_id][channel]

    # Accumulate data into global canvas layers
    img_accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += img_gray_rgb
    mask_accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += tile_mask_rgb
    counts[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += 1

    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1}/{len(files)}')

# 4. Normalize overlapping layers
final_img = np.where(counts > 0, img_accum / counts, 0.0)
final_masks = np.where(counts > 0, mask_accum / counts, 0.0)

# 5. Blend the background grayscale moon with the colored masks
has_mask = np.any(final_masks > 0, axis=-1, keepdims=True)

# Math: Alpha * Mask_Color + (1 - Alpha) * Background_Gray
mosaic = np.where(has_mask, (ALPHA * final_masks) + ((1.0 - ALPHA) * final_img), final_img)
mosaic_u8 = (np.clip(mosaic, 0, 1) * 255).astype(np.uint8)

os.makedirs(OUT_DIR, exist_ok=True)
Image.fromarray(mosaic_u8).save(OUT_FILE)
print(f'Saved {OUT_FILE} ({W}x{H})')

Total tiles: 15935
Grid: 89x179 tiles, Mosaic: 11520x23040
  Processed 1000/15935
  Processed 2000/15935
  Processed 3000/15935
  Processed 4000/15935
  Processed 5000/15935
  Processed 6000/15935
  Processed 7000/15935
  Processed 8000/15935
  Processed 9000/15935
  Processed 10000/15935
  Processed 11000/15935
  Processed 12000/15935
  Processed 13000/15935
  Processed 14000/15935
  Processed 15000/15935
Saved /home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images/moon_mosaic_rgba_overlay.png (23040x11520)


In [6]:
1+1

2

In [7]:
00

0

In [8]:
import numpy as np
from glob import glob
import re, os
from PIL import Image
from scipy.ndimage import label

NPZ_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/data/MR/data/processed/tiles/marius_hills'
OUT_DIR = '/home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images'
OUT_FILE = f'{OUT_DIR}/moon_mosaic_white_bg.png'

files = sorted(glob(f'{NPZ_DIR}/marius_hills_*.npz'))
print(f'Total tiles: {len(files)}')

# 1. Determine mosaic global extent from NPZ row/col values
max_r = max_c = 0
for fpath in files:
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    max_r = max(max_r, ridx)
    max_c = max(max_c, cidx)

TILE_SIZE = 256
STRIDE = 128
H = max_r * STRIDE + TILE_SIZE
W = max_c * STRIDE + TILE_SIZE
print(f'Grid: {max_r+1}x{max_c+1} tiles, Mosaic: {H}x{W}')

# Your exact requested color palette (RGBA format)
mask_colors = [
    (255, 0, 0, 180),      # 0: Red
    (0, 255, 0, 180),      # 1: Green
    (0, 0, 255, 180),      # 2: Blue
    (255, 255, 0, 180),    # 3: Yellow
    (0, 255, 255, 180),    # 4: Cyan
    (255, 0, 255, 180),    # 5: Magenta
    (255, 165, 0, 180),    # 6: Orange
]

# Extract normalized color vectors and a constant alpha scalar
COLORS = [[c[0]/255.0, c[1]/255.0, c[2]/255.0] for c in mask_colors]
ALPHA = mask_colors[0][3] / 255.0  # Uses 180/255 -> ~0.706 opacity

# 2. Setup accumulators for masks and weights
mask_accum = np.zeros((H, W, 3), dtype=np.float64)
counts = np.zeros((H, W, 3), dtype=np.int32)

# 3. Process each tile
for i, fpath in enumerate(files):
    base = os.path.basename(fpath)
    m = re.match(r'marius_hills_r(\d+)_c(\d+)', base)
    if not m:
        continue
    ridx, cidx = int(m.group(1)), int(m.group(2))
    r0, c0 = ridx * STRIDE, cidx * STRIDE

    # Only load masks array
    data = np.load(fpath)
    masks = data['mask']    # Shape: (7, 256, 256)

    # Initialize a mask color canvas for this tile
    tile_mask_rgb = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.float64)

    for class_id in range(7):
        single_mask = masks[class_id] > 0
        th = 100 if class_id == 0 else 300
        lth = 5

        # Label connected components
        labeled, num = label(single_mask)
        filtered_mask = np.zeros_like(single_mask, dtype=np.float64)
        
        for comp_id in range(1, num + 1):
            ys, xs = np.where(labeled == comp_id)
            if len(xs) == 0:
                continue

            box_w = xs.max() - xs.min() + 1
            box_h = ys.max() - ys.min() + 1

            if (box_w < th and box_h < th) and (box_w > lth and box_h > lth):
                filtered_mask[labeled == comp_id] = 1.0

        # Map filtered mask to assigned class colors
        for channel in range(3):
            tile_mask_rgb[..., channel] += filtered_mask * COLORS[class_id][channel]

    # Accumulate data into global canvas layers
    mask_accum[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += tile_mask_rgb
    counts[r0:r0+TILE_SIZE, c0:c0+TILE_SIZE, :] += 1

    if (i + 1) % 1000 == 0:
        print(f'  Processed {i+1}/{len(files)}')

# 4. Normalize overlapping layers
final_masks = np.where(counts > 0, mask_accum / counts, 0.0)

# 5. Build solid White background canvas (all ones)
white_bg = np.ones((H, W, 3), dtype=np.float64)

# 6. Blend masks onto white canvas where components exist
has_mask = np.any(final_masks > 0, axis=-1, keepdims=True)

# Math: Alpha * Mask_Color + (1 - Alpha) * White_Background
mosaic = np.where(has_mask, (ALPHA * final_masks) + ((1.0 - ALPHA) * white_bg), white_bg)
mosaic_u8 = (np.clip(mosaic, 0, 1) * 255).astype(np.uint8)

os.makedirs(OUT_DIR, exist_ok=True)
Image.fromarray(mosaic_u8).save(OUT_FILE)
print(f'Saved {OUT_FILE} ({W}x{H})')

Total tiles: 15935
Grid: 89x179 tiles, Mosaic: 11520x23040
  Processed 1000/15935
  Processed 2000/15935
  Processed 3000/15935
  Processed 4000/15935
  Processed 5000/15935
  Processed 6000/15935
  Processed 7000/15935
  Processed 8000/15935
  Processed 9000/15935
  Processed 10000/15935
  Processed 11000/15935
  Processed 12000/15935
  Processed 13000/15935
  Processed 14000/15935
  Processed 15000/15935
Saved /home/arch/LCPB/2026/code/Computational-Physics-2026/Moon-Recognition/notebooks/images/moon_mosaic_white_bg.png (23040x11520)


In [9]:
1

1